Cell 1: ref_nationality_region (run once)

In [ ]:
from pyspark.sql import Row

regions = {
    "Europe": ["British", "Italian", "French", "German", "Swiss", "Dutch", "Belgium", "Belgian",
               "Irish", "Spanish", "Austrian", "East German", "Russian", "Finnish", "Polish",
               "Portuguese", "Hungarian", "Danish", "Czech", "Liechtensteiner", "Monegasque",
               "Swedish", "Argentine-italian", "American-italian"],
    "North America": ["American", "Canadian", "Mexican"],
    "South America": ["Brazilian", "Chilean", "Argentine", "Uruguayan", "Venezuelan", "Colombian"],
    "Africa": ["South African", "Rhodesian"],
    "Asia": ["Indian", "Chinese", "Japanese", "Malaysian", "Hong Kong", "Indonesian", "Thai"],
    "Oceania": ["Australian", "New Zealand", "New Zealander"],
}

rows = [Row(nationality=n, region=r) for r, ns in regions.items() for n in ns]

(spark.createDataFrame(rows)
      .write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("gold.ref_nationality_region"))

In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
models = [
("dim_races", "gold.dim_races", """
SELECT r.season, r.round, r.race_name, r.race_date,
       c.circuit_name, c.locality, c.country
FROM silver.races r
INNER JOIN silver.circuits c ON r.circuit_id = c.circuit_id
"""),

("dim_constructors", "gold.dim_constructors", """
SELECT c.constructor_id, c.constructor_name, c.nationality,
       n.region AS nationality_region
FROM silver.constructors c
LEFT JOIN gold.ref_nationality_region n ON c.nationality = n.nationality
"""),

("dim_drivers", "gold.dim_drivers", """
SELECT d.driver_id, d.driver_name, d.date_of_birth, d.nationality,
       n.region AS nationality_region
FROM silver.drivers d
LEFT JOIN gold.ref_nationality_region n ON d.nationality = n.nationality
"""),

("fact_session_results", "gold.fact_session_results", """
WITH combined AS (
    SELECT season, round, constructor_id, driver_id, grid_position, completed_laps,
           car_number, points, final_position, final_position_text, status,
           'RACE' AS session_type
    FROM silver.results
    UNION ALL
    SELECT season, round, constructor_id, driver_id, grid_position, completed_laps,
           car_number, points, final_position, final_position_text, status,
           'SPRINT' AS session_type
    FROM silver.sprints
)
SELECT *,
       final_position = 1               AS is_win,
       final_position BETWEEN 1 AND 3   AS is_podium,
       points > 0                       AS has_points
FROM combined
"""),
]

rows = [(m, t, sql.strip(), 1) for m, t, sql in models]

(spark.createDataFrame(rows, "model string, target_table string, sql_text string, is_active int")
      .write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("meta_gold_models"))

display(spark.table("meta_gold_models"))

StatementMeta(, aac688c7-c287-47b8-a0cb-d7136347b30a, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ed948ae2-0ee6-48fa-8898-3326b9d2386c)